In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

target = "converted"
features = df.drop(columns=[target]).copy()

categorical = features.select_dtypes(include=["object", "str"]).columns.tolist()
numerical = features.select_dtypes(exclude=["object", "str"]).columns.tolist()

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

q1_candidates = [
    "lead_score",
    "number_of_courses_viewed",
    "interaction_count",
    "annual_income",
]

q1_scores = {}
for col in q1_candidates:
    score = roc_auc_score(y_train, df_train[col])
    if score < 0.5:
        score = roc_auc_score(y_train, -df_train[col])
    q1_scores[col] = score
    print(f"AUC for {col}: {score:.4f}")

best_feature = max(q1_scores, key=q1_scores.get)
print("Q1 Highest AUC feature:", best_feature)

all_features = categorical + numerical

dv = DictVectorizer(sparse=False)
train_dict = df_train[all_features].to_dict(orient="records")
X_train = dv.fit_transform(train_dict)

val_dict = df_val[all_features].to_dict(orient="records")
X_val = dv.transform(val_dict)

model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, y_pred)
print("Q2 Validation AUC:", round(val_auc, 3))

thresholds = np.arange(0.0, 1.01, 0.01)

scores_pr = []
f1_scores = []

for t in thresholds:
    actual_pos = (y_val == 1)
    actual_neg = (y_val == 0)

    pred_pos = (y_pred >= t)
    pred_neg = (y_pred < t)

    tp = (pred_pos & actual_pos).sum()
    tn = (pred_neg & actual_neg).sum()
    fp = (pred_pos & actual_neg).sum()
    fn = (pred_neg & actual_pos).sum()

    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0

    if p > 0 or r > 0:
        scores_pr.append((abs(p - r), t, p, r))

    f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
    f1_scores.append((f1, t))

best_intersection = min(scores_pr, key=lambda x: x[0])[1]
print("Q3 Precision and recall intersect at:", round(best_intersection, 2))

best_f1_t = max(f1_scores, key=lambda x: x[0])[1]
print("Q4 Highest F1 threshold:", round(best_f1_t, 2))

def train(df_train_fold, y_train_fold, C=1.0):
    dicts = df_train_fold[all_features].to_dict(orient="records")
    dv_fold = DictVectorizer(sparse=False)
    X_train_fold = dv_fold.fit_transform(dicts)

    model_fold = LogisticRegression(solver="liblinear", C=C, max_iter=1000)
    model_fold.fit(X_train_fold, y_train_fold)

    return dv_fold, model_fold

def predict(df_val_fold, dv_fold, model_fold):
    dicts = df_val_fold[all_features].to_dict(orient="records")
    X_val_fold = dv_fold.transform(dicts)
    return model_fold.predict_proba(X_val_fold)[:, 1]

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
cv_scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_t = df_full_train.iloc[train_idx]
    df_v = df_full_train.iloc[val_idx]

    y_t = df_t[target].values
    y_v = df_v[target].values

    dv_fold, model_fold = train(df_t, y_t, C=1.0)
    y_pred_v = predict(df_v, dv_fold, model_fold)

    auc = roc_auc_score(y_v, y_pred_v)
    cv_scores.append(auc)

print("Q5 5-Fold CV std:", round(np.std(cv_scores), 3))

c_values = [0.000001, 0.001, 1]
c_results = {}

for C in c_values:
    scores = []

    for train_idx, val_idx in kfold.split(df_full_train):
        df_t = df_full_train.iloc[train_idx]
        df_v = df_full_train.iloc[val_idx]

        y_t = df_t[target].values
        y_v = df_v[target].values

        dv_fold, model_fold = train(df_t, y_t, C=C)
        y_pred_v = predict(df_v, dv_fold, model_fold)

        auc = roc_auc_score(y_v, y_pred_v)
        scores.append(auc)

    mean_auc = round(np.mean(scores), 3)
    std_auc = round(np.std(scores), 3)
    c_results[C] = (mean_auc, std_auc)
    print(f"C={C} -> Mean: {mean_auc}, Std: {std_auc}")

best_c = max(c_values, key=lambda c: (c_results[c][0], -c_results[c][1], -c))
print("Q6 Best C parameter:", best_c)

AUC for lead_score: 0.7882
AUC for number_of_courses_viewed: 0.7230
AUC for interaction_count: 0.7649
AUC for annual_income: 0.6082
Q1 Highest AUC feature: lead_score
Q2 Validation AUC: 0.732
Q3 Precision and recall intersect at: 0.63
Q4 Highest F1 threshold: 0.41
Q5 5-Fold CV std: 0.007
C=1e-06 -> Mean: 0.617, Std: 0.019
C=0.001 -> Mean: 0.749, Std: 0.01
C=1 -> Mean: 0.732, Std: 0.007
Q6 Best C parameter: 0.001
